# 5 Supervised Models

## 5.1 Imports 

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 5.2 Preprocessing

In [ ]:
TARGET = "games_missed"

NUMERIC_FEATURES = [
    "age",
    "career_games",
    "career_minutes",
    "avg_minutes",
    "usage_rate",
    "distance_traveled",
    "avg_speed",
    "drives_per_game",
    "back_to_back_games",
    "games_missed_last_3_seasons"
]

CATEGORICAL_FEATURES = [
    "position"
]


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES)
    ]
)


In [ ]:
def split_train_test_by_season(df, test_season):
    train_df = df[df["season"] != test_season]
    test_df = df[df["season"] == test_season]

    X_train = train_df.drop(columns=[TARGET])
    y_train = train_df[TARGET]

    X_test = test_df.drop(columns=[TARGET])
    y_test = test_df[TARGET]

    return X_train, X_test, y_train, y_test


## 5.3 Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

linreg_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])


## 5.4 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ))
])


## 5.5 XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ))
])


## 5.6 Training and Evaluation

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)

    return mae, rmse


## Example

In [ ]:
X_train, X_test, y_train, y_test = split_train_test_by_season(
    df, test_season="2024-25"
)

models = {
    "Linear Regression": linreg_pipeline,
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline
}

results = {}

for name, model in models.items():
    mae, rmse = evaluate_model(model, X_train, y_train, X_test, y_test)
    results[name] = {"MAE": mae, "RMSE": rmse}

pd.DataFrame(results).T
